In [4]:
import pandas as pd
from scipy.stats import t

measure = pd.read_csv("/Users/fara89/Desktop/measure_over_time.csv")

def trial_uplift_analysis(measure_df, trial_store, control_store):

    df = measure_df[measure_df["store_nbr"].isin([trial_store, control_store])].copy()

    pretrial = df[df["yearmonth"] < 201902]
    trial = df[(df["yearmonth"] >= 201902) & (df["yearmonth"] <= 201904)]

    def analyse_metric(metric):

        trial_pre = pretrial[pretrial["store_nbr"] == trial_store][["yearmonth", metric]].rename(
            columns={metric: "trial"}
        )

        control_pre = pretrial[pretrial["store_nbr"] == control_store][["yearmonth", metric]].rename(
            columns={metric: "control"}
        )

        pre = trial_pre.merge(control_pre, on="yearmonth")

        scaling_factor = pre["trial"].sum() / pre["control"].sum()

        pre["scaled_control"] = pre["control"] * scaling_factor

        pre["pct_diff"] = abs(pre["trial"] - pre["scaled_control"]) / pre["scaled_control"]

        mean_pre = pre["pct_diff"].mean()
        std_pre = pre["pct_diff"].std()

        trial_trial = trial[trial["store_nbr"] == trial_store][["yearmonth", metric]].rename(
            columns={metric: "trial"}
        )

        control_trial = trial[trial["store_nbr"] == control_store][["yearmonth", metric]].rename(
            columns={metric: "control"}
        )

        trial_df = trial_trial.merge(control_trial, on="yearmonth")

        trial_df["scaled_control"] = trial_df["control"] * scaling_factor

        trial_df["pct_diff"] = abs(trial_df["trial"] - trial_df["scaled_control"]) / trial_df["scaled_control"]

        dfree = len(pre) - 1
        critical = t.ppf(0.95, df=dfree)

        trial_df["t_value"] = (trial_df["pct_diff"] - mean_pre) / std_pre

        trial_df["significant"] = trial_df["t_value"] > critical

        return trial_df

    sales_result = analyse_metric("tot_sales")
    cust_result = analyse_metric("n_customers")

    return sales_result, cust_result

In [5]:
sales77, cust77 = trial_uplift_analysis(measure, 77, 233)

print("Sales result")
print(sales77)

print("Customer result")
print(cust77)

Sales result
   yearmonth  trial  control  scaled_control  pct_diff   t_value  significant
0     201902  211.6    220.7      229.473346  0.077889 -0.326981        False
1     201903  255.1    180.6      187.779277  0.358510  4.082600         True
2     201904  258.1    144.2      149.932291  0.721444  9.785611         True
Customer result
   yearmonth  trial  control  scaled_control  pct_diff    t_value  significant
0     201902     40       42       42.913043  0.067882   1.162692        False
1     201903     46       35       35.760870  0.286322   8.433462         True
2     201904     47       27       27.586957  0.703704  22.326015         True


In [6]:
sales86, cust86 = trial_uplift_analysis(measure, 86, 155)
print("Sales result86")
print(sales86)

print("Customer result 86")
print(cust86)

Sales result86
   yearmonth  trial  control  scaled_control  pct_diff    t_value  significant
0     201902  872.8    850.8      827.019610  0.055356   1.127017        False
1     201903  945.4    767.0      745.561872  0.268037  11.280851         True
2     201904  804.0    800.4      778.028321  0.033381   0.077910        False
Customer result 86
   yearmonth  trial  control  scaled_control  pct_diff   t_value  significant
0     201902    105       92       92.276276  0.137887  6.863935         True
1     201903    108       91       91.273273  0.183260  9.695696         True
2     201904     99       93       93.279279  0.061329  2.085833         True


In [7]:
sales88, cust88 = trial_uplift_analysis(measure, 88, 237)
print("Sales result88")
print(sales88)

print("Customer result88")
print(cust88)

Sales result88
   yearmonth   trial  control  scaled_control  pct_diff   t_value  significant
0     201902  1339.6   1313.0     1300.879003  0.029765 -0.496849        False
1     201903  1467.0   1177.6     1166.728952  0.257361  4.140575         True
2     201904  1317.0   1153.6     1142.950509  0.152281  1.999488         True
Customer result88
   yearmonth  trial  control  scaled_control  pct_diff   t_value  significant
0     201902    122      119      118.441315  0.030046  0.540110        False
1     201903    133      116      115.455399  0.151960  7.345100         True
2     201904    119      116      115.455399  0.030701  0.576674        False
